# 反思（Reflection）

## 反思模式概述
在之前的章节中，我们探讨了基本的智能体模式：顺序执行链、动态路径选择路由以及任务并行化。这些模式让智能体能够高效灵活地完成复杂任务。然而，当面对复杂工作流时，智能体的初始输出或计划可能并非最优、准确或完整，这时就需要引入**反思**模式。

反思模式涉及智能体评估自身的工作、输出或内部状态，并利用此评估来改进性能或调整响应。它是一种自我修正或自我提升的形式，通过反馈、内部批评或与期望标准比较，智能体迭代优化输出或方式。

反思通过引入反馈循环而非简单的顺序链，智能体不仅生成输出，还会检查输出，识别问题或改进空间，利用洞察生成更优版本或调整未来动作。

反思流程通常包含：
1. **执行（Execution）**：智能体执行任务或生成初始输出。
2. **评估/批评（Evaluation/Critique）**：智能体或另一个LLM对上一步结果进行分析，检查事实准确性、连贯性、风格、完整性及指令遵守等。
3. **反思/改进（Reflection/Refinement）**：基于评估结果，智能体决定如何提升，可能生成精炼输出、调整参数或修改整体计划。
4. **迭代（Iteration，常见但可选）**：执行改进后的方案，并反复进行反思直到结果满意或达到停止条件。

## 反思模式的实现
反思通常采用“生产者-批评者”模型：生产者负责初始生成，批评者负责评估反馈。使用两个专门化智能体（或两个独立的LLM调用），分别作为生产者和评价者，可以获得更鲁棒、不偏见的结果。

1. 生产者智能体：专注于执行任务，生成内容，如写代码、博客或计划。
2. 批评者智能体：评估生产者输出，基于具体标准如准确性、代码质量、风格或完整性提出结构化反馈。

该分工防止智能体自身认知偏见，批评者以新视角审视产出，将反馈传回生产者，迭代生成更优版本。LangChain和ADK等开发工具已支持这种双智能体设计。

## 反思模式的应用
实现反思需构建反馈循环，可以通过代码中的迭代循环或支持状态管理、条件转移的框架来实现。单一评估-改进步骤较简单，复杂任务则需要多次反馈循环实现更完善的反思。

反思模式对生成高质量输出、处理细微任务、展现自我觉察和适应性至关重要。它让智能体不再是被动执行者，而成为主动调整、解决问题的智慧系统。

## 与目标设定和监控的关系
反思与目标设定和监控（见第11章）紧密相关：目标定下基准，监控跟踪进展，反思基于反馈调整策略。这使智能体从被动执行者变为主动适应达到目标的系统。

## 记忆与反思的协同提升
当大型语言模型（LLM）具备对话记忆时，反思效果显著增强。对话历史为评估提供背景，智能体基于过往交互、用户反馈和目标逐步学习、进化。否则，反思是逐步累积的过程，记忆积累带来更智能、更具上下文感知的改进。

## 实际应用与使用场景总结

反思模式在输出质量、准确性或遵守复杂约束条件时非常重要，具体应用如下：

1. **创意写作与内容生成**
   - **使用场景**：代理撰写博客文章。
   - **反思**：生成草稿，针对内容流畅性、语气、清晰度进行批判性审查，再根据反馈重写，直至满足质量标准。
   - **益处**：产出更打磨且高效的内容。

2. **代码生成与调试**
   - **使用场景**：代理编写 Python 函数。
   - **反思**：编写初始代码，运行测试或静态分析，识别错误或低效部分，然后根据结果修改代码。
   - **益处**：生成更健壮、功能完善的代码。

3. **复杂问题解决**
   - **使用场景**：代理解决逻辑难题。
   - **反思**：提出步骤，评估是否接近解答，避免矛盾，必要时回溯或选择不同路径。
   - **益处**：提升代理解决复杂问题的能力。

4. **摘要与信息综合**
   - **使用场景**：代理总结长文档。
   - **反思**：生成初始摘要，比对关键点，完善补充遗漏信息，提高准确度。
   - **益处**：创建更准确、全面的摘要。

5. **规划与策略**
   - **使用场景**：代理规划一系列行动以达成目标。
   - **反思**：生成计划，模拟执行或评估可行性，根据约束调整计划。
   - **益处**：开发更有效、现实的计划。

6. **会话代理**
   - **使用场景**：客服聊天机器人。
   - **反思**：在用户回复后，回顾对话历史和最新输入，确保连贯性和准确响应。
   - **益处**：实现更自然、高效的对话。


反思为智能体系统增加了元认知层，使其能够从自身输出和过程学习，提升智能性、可靠性及高质量结果的产生。

## 实战示例


### 使用Langchain

- 首先设置环境，加载API密钥，并初始化一个强大的语言模型（如GPT-4），设置较低温度以得到集中输出。
- 主要任务是通过一个提示，让语言模型生成计算阶乘的Python函数代码，要求包括：
  - 函数文档字符串（docstrings）
  - 边界情况处理（如0的阶乘）
  - 负数输入的错误处理
- 通过`run_reflection_loop`函数协调迭代优化过程：
  - 在第一次迭代中，语言模型根据任务提示生成初始代码。
  - 后续迭代中，语言模型根据上一步的反馈进行代码优化。
  - 设定一个“反思者”角色（也是语言模型），使用不同的系统提示，作为高级软件工程师对生成的代码进行审查，检查是否符合任务要求。
  - 使用项目符号给出问题清单或用"CODE_IS_PERFECT"表示无问题。
  - 循环直到代码完美或达到最大迭代次数。
- 会话历史会被维护并传递给语言模型，确保上下文连贯。
- 最后脚本会打印经过多次迭代优化后的最终代码版本。


In [1]:
import dotenv
dotenv.load_dotenv()


True

In [6]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableBranch
from langchain_core.messages import SystemMessage, HumanMessage
from pprint import pprint

In [7]:
try:
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
    pprint(f"Language model initialized: {llm.model}")
except Exception as e:
    pprint(f"Error initializing language model: {e}")
    llm = None

'Language model initialized: gemini-2.5-flash'


In [8]:
def run_reflection_loop():
    """
    Demonstrates a multi-step AI reflection loop to progressively improve a Python function.
    展示了通过多步骤反思循环，逐步改进 Python 函数的方法。
    """

    # --- The Core Task ---
    # --- 核心任务的提示词 ---
    task_prompt = """
    Your task is to create a Python function named `calculate_factorial`.
    This function should do the following:
    1.  Accept a single integer `n` as input.
    2.  Calculate its factorial (n!).
    3.  Include a clear docstring explaining what the function does.
    4.  Handle edge cases: The factorial of 0 is 1.
    5.  Handle invalid input: Raise a ValueError if the input is a negative number.
    """

    # --- The Reflection Loop ---
    # --- 反思循环 ---
    max_iterations = 3
    current_code = ""
    # We will build a conversation history to provide context in each step.
    # 构建对话历史，为每一步提供必要的上下文信息。
    message_history = [HumanMessage(content=task_prompt)]

    for i in range(max_iterations):
        print("\n" + "="*25 + f" REFLECTION LOOP: ITERATION {i + 1} " + "="*25)

        # --- 1. GENERATE / REFINE STAGE ---
        # In the first iteration, it generates. In subsequent iterations, it refines.
        # 在第一次迭代时，生成初始代码；在后续迭代时，基于上一步的反馈优化代码。
        if i == 0:
            print("\n>>> STAGE 1: GENERATING initial code...")
            # The first message is just the task prompt.
            # 第一次迭代时，只需要任务提示词。
            response = llm.invoke(message_history)
            current_code = response.content
        else:
            print("\n>>> STAGE 1: REFINING code based on previous critique...")
            # The message history now contains the task, the last code, and the last critique.
            # We instruct the model to apply the critiques.
            # 后续迭代时，除了任务提示词，还包含上一步的代码和反馈。
            # 然后要求模型根据反馈意见优化代码。
            message_history.append(HumanMessage(content="Please refine the code using the critiques provided."))
            response = llm.invoke(message_history)
            current_code = response.content

        print("\n--- Generated Code (v" + str(i + 1) + ") ---\n" + current_code)
        message_history.append(response) # Add the generated code to history

        # --- 2. REFLECT STAGE ---
        # --- 反思阶段 ---
        print("\n>>> STAGE 2: REFLECTING on the generated code...")

        # Create a specific prompt for the reflector agent.
        # This asks the model to act as a senior code reviewer.
        # 创建一个特定的提示词，要求模型扮演高级软件工程师的角色，对代码进行仔细的审查。
        reflector_prompt = [
            SystemMessage(content="""
                You are a senior software engineer and an expert in Python.
                Your role is to perform a meticulous code review.
                Critically evaluate the provided Python code based on the original task requirements.
                Look for bugs, style issues, missing edge cases, and areas for improvement.
                If the code is perfect and meets all requirements, respond with the single phrase 'CODE_IS_PERFECT'.
                Otherwise, provide a bulleted list of your critiques.
            """),
            HumanMessage(content=f"Original Task:\n{task_prompt}\n\nCode to Review:\n{current_code}")
        ]

        critique_response = llm.invoke(reflector_prompt)
        critique = critique_response.content

        # --- 3. STOPPING CONDITION ---
        # 如果代码完美符合要求，则结束反思循环。
        if "CODE_IS_PERFECT" in critique:
            print("\n--- Critique ---\nNo further critiques found. The code is satisfactory.")
            break

        print("\n--- Critique ---\n" + critique)
        # Add the critique to the history for the next refinement loop.
        message_history.append(HumanMessage(content=f"Critique of the previous code:\n{critique}"))

    print("\n" + "="*30 + " FINAL RESULT " + "="*30)
    print("\nFinal refined code after the reflection process:\n")
    print(current_code)

In [9]:
run_reflection_loop()


========================= REFLECTION LOOP: ITERATION 1 =========================

>>> STAGE 1: GENERATING initial code...

--- Generated Code (v1) ---
```python
def calculate_factorial(n: int) -> int:
    """
    Calculates the factorial of a non-negative integer.

    The factorial of a non-negative integer n, denoted by n!, is the product
    of all positive integers less than or equal to n.

    Special cases:
    - The factorial of 0 is 1.
    - Factorial is not defined for negative numbers.

    Args:
        n (int): The non-negative integer for which to calculate the factorial.

    Returns:
        int: The factorial of n.

    Raises:
        ValueError: If the input `n` is a negative number.
    """
    if not isinstance(n, int):
        raise TypeError("Input must be an integer.")
        
    if n < 0:
        raise ValueError("Factorial is not defined for negative numbers.")
    elif n == 0:
        return 1
    else:
        factorial_result = 1
        for i in range(1,

### 使用 Google ADK

这段代码演示了如何在 Google ADK 中使用顺序代理（sequential agent）流水线来生成和审核文本。内容要点如下：

- 定义了两个 LlmAgent 实例：生成器（generator）和审核员（reviewer）。
- 生成器的任务是根据指定主题，写一段简短且有信息量的初稿段落，并将结果保存到状态键 `draft_text`。
- 审核员充当事实核查员，读取生成器输出的文本（`draft_text`），验证其准确性。
- 审核员输出一个结构化的字典，包含两个键：`status`（"ACCURATE"或"INACCURATE"）和`reasoning`（状态的解释）。
- 审核结果保存到状态键 `review_output`。
- 使用 `SequentialAgent` 创建名为 `review_pipeline` 的流水线，控制两个代理的执行顺序：先生成器，再审核员。
- 整体执行流程为生成器产生文本并保存状态，然后审核员读取文本，执行事实核查，并保存审核结果。
- 该流水线使内容创作和审核分开进行，更加结构化。
- 另外，提到还可以用 ADK 的 LoopAgent 实现类似功能。



- Reflection 模式虽然能显著提升输出质量，但带来了重要的权衡：
  - 迭代过程可能导致更高的成本和延迟，因为每次迭代需调用新的 LLM 请求，可能不适合对时间敏感的应用。
  - 该模式内存密集，每次迭代都会扩展对话历史，包括初始输出、批注和后续改进内容。

In [ ]:
from google.adk.agents import SequentialAgent, LlmAgent


# 第一个智能体生成初始草稿。
generator = LlmAgent(
    name="DraftWriter",
    description="Generates initial draft content on a given subject.",
    instruction="Write a short, informative paragraph about the user's subject.",
    output_key="draft_text" # The output is saved to this state key.
)


# 第二个智能体评审第一个智能体的草稿。
reviewer = LlmAgent(
    name="FactChecker",
    description="Reviews a given text for factual accuracy and provides a structured critique.",
    instruction="""
    You are a meticulous fact-checker.
    1. Read the text provided in the state key 'draft_text'.
    2. Carefully verify the factual accuracy of all claims.
    3. Your final output must be a dictionary containing two keys:
       - "status": A string, either "ACCURATE" or "INACCURATE".
       - "reasoning": A string providing a clear explanation for your status, citing specific issues if any are found.
    """,
    output_key="review_output" # The structured dictionary is saved here.
)


# 确保生成者在评论者之前运行。
review_pipeline = SequentialAgent(
    name="WriteAndReview_Pipeline",
    sub_agents=[generator, reviewer]
)

### 使用 Datasurfer

- 这段代码展示了如何使用 Datasurfer 库中的 Coder 和 CodeReviewer 智能体进行反思式代码生成和评审。
- 首先创建了两个智能体实例：一个是代码生成者（Coder），另一个是代码评审者（CodeReviewer）。评审者被赋予了一个描述性的提示，指导其评审标准和方式。
- 代码生成者接收一个任务提示，开始生成代码。    
- 生成者和评审者之间进行多轮交互，代码生成者根据评审者的反馈不断改进代码，最多允许3轮反思和改进。
- 这种反思式的交互使得代码生成过程更加迭代和完善，最终产出更高质量的代码。

In [16]:
from datasurfer.lib_llm.expertise import Coder, CodeReviewer


In [ ]:
# --- The Core Task ---
# --- 核心任务的提示词 ---
task_prompt = """
Your task is to create a Python function named `calculate_factorial`.
This function should do the following:
1.  Accept a single integer `n` as input.
2.  Calculate its factorial (n!).
3.  Include a clear docstring explaining what the function does.
4.  Handle edge cases: The factorial of 0 is 1.
5.  Handle invalid input: Raise a ValueError if the input is a negative number.
"""

reflector_prompt = """
You are a senior software engineer and an expert in Python.
Your role is to perform a meticulous code review.
Critically evaluate the provided Python code based on the original task requirements.
Look for bugs, style issues, missing edge cases, and areas for improvement.
If the code is perfect and meets all requirements, respond with the single phrase 'CODE_IS_PERFECT'.
Otherwise, provide a bulleted list of your critiques.
"""



In [28]:
# 创建两个智能体：一个是代码生成者，另一个是代码评审者。
coder = Coder(name='Coder').golive('AA')
reflector = CodeReviewer(name='Reflector', description=reflector_prompt).golive('AA')

In [ ]:
# 3轮反思和改进
coder.receive_message(task_prompt)
for _ in range(3):  # Allow up to 3 iterations of reflection and refinement
    coder.reply().send_message(reflector)
    reflector.reply().send_message(coder)

Coder:

    ```python
    def calculate_factorial(n: int) -> int:
        """
        Calculate the factorial of a non-negative integer n.

        The factorial of n (denoted as n!) is the product of all positive integers less than or
    equal to n.
        By definition, the factorial of 0 is 1.

        Parameters:
        n (int): A non-negative integer whose factorial is to be calculated.

        Returns:
        int: The factorial of the given number n.

        Raises:
        ValueError: If n is a negative integer.
        """
        if n < 0:
            raise ValueError("Input must be a non-negative integer.")
        if n == 0:
            return 1

        factorial = 1
        for i in range(1, n + 1):
            factorial *= i
        return factorial
    ```

--------------------------------------------------------------------------------------------------------

Reflector:

    - The function correctly verifies that `n` is non-negative and raises a `ValueError` for
